# 2026/27-Time Series Lab
Note that this lab has three levels: basic, regular and advanced.\
Doing the **basic** part earns you a grade of 5.5-6.0.\
Doing the **regular** part earns you a max grade of 8.\
Doing the **advanced** part earns you a max grade of 10.0.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.

**Group No:**

**Student Name**:

**Student Name**:

## Necessary Imports
The following import shall be sufficient.

In [ ]:
import pandas as pd # for data manipulation
import numpy as np # numerical work
import matplotlib.pyplot as plt # plotting
import seaborn as sns # stats plots
import statsmodels.formula.api as smf # most time series formulas
import statsmodels.api as sm # general package call


## Exercise 0
(basic)Write one line about each of the package imported above that describe their functionality or why they are imported.

pandas is a spreadsheet for python, it exposes a DataFrame: an object for tabular data manipulation.
numpy is a low-level numeric manipulation: its a fast library writen in C for manipulating arrays. its also the building block of pandas.
matplotlib is a plotting tool, responsible to generate highly customizable but verbose plots.
seaborn is a wrapper over matplotlib that expose better plot configurations for common workloads with clean syntax.
statsmodel contains the statistical models for time series prediction and analysis.

!-- #endregion -->

<!-- #region id="3znLCsERgyVL" -->
## Exercise 1
Use python to simulate and plot some data from simple ARIMA models. The exercise is taken from the book https://otexts.com/fpp2/arima-exercises.html. You are encouraged to read the corresponding chapter to prepare for the examination.  




**a.(basic)** Use the following Python code to generate data from an AR(1) model with  $𝜙_1$=0.6  and  $𝜎_2$=1 . The process starts with  $𝑦_1$=0 . (You just need to run the cell to generate data).

In [ ]:
np.random.seed(25)# fix the random seed
mu, sigma = 0, 1 # mean and standard deviation
y = np.zeros(100)
e = np.random.normal(mu, sigma, 100)
phi1=0.6
for i in range(2,100):
    y[i] = phi1*y[i-1] + e[i]

**b.(basic)** Produce a time plot for the series. How does the plot change as you change $\phi_1$?. It is sufficient to produce a plots for different values of $\phi_1$.

In [ ]:
def simulate_ar1(phi1, n=100, seed=25):
    np.random.seed(seed)
    e = np.random.normal(0, 1, n)
    y = np.zeros(n)
    for i in range(2, n):
        y[i] = phi1 * y[i-1] + e[i]
    return y

phis = [0.3, 0.6, 0.9, -0.6]
fig, axes = plt.subplots(len(phis), 1, figsize=(9, 9), sharex=True)
for ax, phi in zip(axes, phis):
    ax.plot(simulate_ar1(phi))
    ax.set_title(f"AR(1), $\\phi_1$={phi}")
plt.tight_layout()
plt.show()

**Answer:** As $\phi_1$ grows towards 1, the series becomes smoother and more persistent, shocks decay slowly, so the series wanders in long runs above or below zero (higher autocorrelation). As $\phi_1 \to 0$, the series looks more like white noise (no memory of the past). A negative $\phi_1$ (e.g. -0.6) makes the series oscillate rapidly from one observation to the next, alternating sign.

**c.(basic)** Write your own code to generate data from an MA(1) model with $\theta_1=0.6$ and $\sigma_2=1$. Plot the results also.

In [ ]:
np.random.seed(25)
n = 100
e = np.random.normal(0, 1, n)
theta1 = 0.6
y_ma1 = np.zeros(n)
for i in range(1, n):
    y_ma1[i] = e[i] + theta1 * e[i-1]

plt.figure(figsize=(9, 3))
plt.plot(y_ma1)
plt.title(f"MA(1), $\\theta_1$={theta1}")
plt.show()

**d.(Regular)** Generate data from an ARMA(1,1) model with $\phi_1 = 0.6$ and $\theta_1=0.6$ and $\sigma_2=1$.


In [ ]:
np.random.seed(25)
n = 100
e = np.random.normal(0, 1, n)
phi1, theta1 = 0.6, 0.6
y_arma11 = np.zeros(n)
for i in range(1, n):
    y_arma11[i] = phi1 * y_arma11[i-1] + e[i] + theta1 * e[i-1]

plt.figure(figsize=(9, 3))
plt.plot(y_arma11)
plt.title(f"ARMA(1,1), $\\phi_1$={phi1}, $\\theta_1$={theta1}")
plt.show()

**e.(Regular)** Generate data from an AR(2) model with $\phi_1=-0.8$, and $\phi_2=0.3$ and $\sigma_2=1$. (Note that these parameters will give a non-stationary series.)


In [ ]:
np.random.seed(25)
n = 100
e = np.random.normal(0, 1, n)
phi1, phi2 = -0.8, 0.3
y_ar2 = np.zeros(n)
for i in range(2, n):
    y_ar2[i] = phi1 * y_ar2[i-1] + phi2 * y_ar2[i-2] + e[i]

plt.figure(figsize=(9, 3))
plt.plot(y_ar2)
plt.title(f"AR(2), $\\phi_1$={phi1}, $\\phi_2$={phi2}")
plt.show()

**f.(Regular)**  Graph the latter two series and compare them.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(y_arma11)
axes[0].set_title("ARMA(1,1): stationary, bounded fluctuations")
axes[1].plot(y_ar2)
axes[1].set_title("AR(2) with $\\phi_1$=-0.8, $\\phi_2$=0.3: non-stationary, diverges")
plt.tight_layout()
plt.show()

**g.(Regular)** How can you verify that the series in $e$ is indeed non-stationary?

**Answer:** An AR($p$) process is stationary only if all roots of its characteristic polynomial $1-\phi_1 z - \phi_2 z^2=0$ lie outside the unit circle (modulus > 1). For $\phi_1=-0.8$, $\phi_2=0.3$ one of the roots has modulus < 1, so the series is non-stationary, this matches the plot above, where the series diverges instead of fluctuating around a fixed mean/variance. 

In [ ]:
# Characteristic polynomial 1 - phi1*z - phi2*z^2 = 0
roots = np.roots([-phi2, -phi1, 1])
print("Characteristic roots:", roots, "\nmoduli:", np.abs(roots))
print("Non-stationary:", bool((np.abs(roots) <= 1).any()))


## Exercise 2

Consider usmelec, the total net generation of electricity (in billion kilowatt hours) by the U.S. electric industry (monthly for the period January 1973 – June 2013). In general there are two peaks per year: in mid-summer and mid-winter.

The exercise is taken from the book https://otexts.com/fpp2/arima-exercises.html. You are encouraged to read the corresponding chapter to prepare for the examination.


In [ ]:
# Read data
from pathlib import Path
datapath = Path.cwd() / "dt-lab-3" / "databank" / "usmelec.csv"
usmelec_data = pd.read_csv(datapath)
usmelec_data.head()# Inspect first few values.

Note that index is just the column header. For later convinience we change the date format to yyyy-mm-dd. And since it is a time series let us set its index as the date.

In [ ]:
usmelec_data['index'] = pd.to_datetime(usmelec_data['index'], format='%Y %b') #change format to datetime
usmelec_data.set_index('index',inplace=True)# setting index.
usmelec_data.head()

**a. (Basic)** Create line plot for the above time series. Do you observe a trend or seasonality in this plot?

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(usmelec_data.index, usmelec_data['value'])
plt.title("US net electricity generation")
plt.xlabel("Date")
plt.ylabel("Billion kWh")
plt.show()

**Answer:** There is a clear long-run upward trend (electricity demand grows over the 40 years), and a strong yearly seasonal pattern layered on top of it, as noted in the exercise description, there are two peaks per year (mid-summer and mid-winter, from air-conditioning and heating demand). The amplitude of the seasonal fluctuations also grows as the series level grows, hinting that a log transform will be useful.

**b.(Basic)** Read the documentation about creating [KDE](https://en.wikipedia.org/wiki/Kernel_density_estimation) plot  [here](https://seaborn.pydata.org/generated/seaborn.kdeplot.html). Create density plot for your time series.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(usmelec_data['value'], fill=True)
plt.title("Density of monthly electricity generation")
plt.xlabel("Billion kWh")
plt.show()

 **c.(Basic)**	Create Box plot and violin plot for monthly generation. What seasonal pattern you can deduce from these plots.

In [ ]:
usmelec_data['month'] = usmelec_data.index.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='month', y='value', data=usmelec_data, ax=axes[0])
axes[0].set_title("Boxplot by month")
sns.violinplot(x='month', y='value', data=usmelec_data, ax=axes[1])
axes[1].set_title("Violin plot by month")
plt.tight_layout()
plt.show()

**Answer:** Generation is highest and most spread out in January/February (winter heating) and July/August (summer cooling), and lowest in the shoulder months (April/May and October/November). This bimodal, twin-peak seasonal pattern confirms the description of the series and matches the strong yearly seasonality seen in the line plot.

**d.(Basic)** Create [PACF](https://www.statsmodels.org/dev/generated/statsmodels.tsa.stattools.pacf.html?highlight=pacf)

What you can conclude from this visulization.

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(usmelec_data['value'], lags=36, ax=ax)
plt.title("PACF of usmelec")
plt.show()

**Answer:** The PACF has a huge spike near 1 at lag 1 (typical of a strongly persistent, non-stationary/trending series) and further significant spikes around lags 12-13, reflecting the yearly seasonal cycle. This pattern (slow decay plus seasonal spikes) suggests the raw series is non-stationary and that both a non-seasonal and a seasonal (lag-12) AR/differencing component will be needed before fitting an ARIMA model.

**e.(Basic)**	Examine the 12-month moving average and variance of this series to see what kind of trend is involved.

In [ ]:
ma12 = usmelec_data['value'].rolling(window=12).mean()
var12 = usmelec_data['value'].rolling(window=12).var()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(usmelec_data.index, usmelec_data['value'], alpha=0.4, label='original')
axes[0].plot(usmelec_data.index, ma12, color='red', label='12-month moving average')
axes[0].set_title("12-month moving average")
axes[0].legend()
axes[1].plot(usmelec_data.index, var12, color='green')
axes[1].set_title("12-month rolling variance")
plt.tight_layout()
plt.show()

**Answer:** The 12-month moving average shows a steady, roughly monotonic upward trend (not perfectly linear, it flattens somewhat after ~2008). The rolling variance also increases over time, roughly tracking the level of the series. Since variance grows with the level, this points to a multiplicative trend/seasonality, a log transform should stabilise it.

**f.(Regular)**  Substantiate here breifly why your choosen transformation is appropriate?

**Answer:** As shown in (e), the rolling variance increases roughly in proportion to the series level, a classic sign of multiplicative seasonality/heteroscedasticity. Taking $\log(\text{value})$ turns the multiplicative structure into an additive one and stabilises the variance, which is required for ARIMA models that assume homoscedastic (constant-variance) residuals. All differencing and modelling below is therefore done on the log-transformed series.

**g.(Regular)**	Are the data stationary? If not, find an appropriate differencing which yields stationary data. Most time second order differencing is sufficient.

In [ ]:
log_usmelec = np.log(usmelec_data['value'])
diff_seasonal = log_usmelec.diff(12).dropna()          # remove yearly seasonality
diff_seasonal_first = diff_seasonal.diff().dropna()     # + remove remaining trend

fig, axes = plt.subplots(3, 1, figsize=(11, 9))
axes[0].plot(log_usmelec)
axes[0].set_title("log(usmelec)")
axes[1].plot(diff_seasonal)
axes[1].set_title("Seasonally differenced (lag 12)")
axes[2].plot(diff_seasonal_first)
axes[2].set_title("Seasonally + first differenced")
plt.tight_layout()
plt.show()

Substantiate that your choosen difference is appropriate?

**Hint:** [Augmented Dickey–Fuller](https://www.statsmodels.org/dev/generated/statsmodels.tsa.stattools.adfuller.html) test can be used to test stationarity.

If you need code add a code block yourself.

In [ ]:
from statsmodels.tsa.stattools import adfuller

for name, series in [("log level", log_usmelec),
                      ("seasonal diff (lag 12)", diff_seasonal),
                      ("seasonal + first diff", diff_seasonal_first)]:
    stat, pvalue, *_ = adfuller(series)
    print(f"{name:28s}: ADF stat={stat:7.3f}, p-value={pvalue:.4g}")

**Answer:** The ADF test on the log level cannot reject the unit-root null (p≈0.23), confirming non-stationarity. After a lag-12 seasonal difference, the p-value already drops well below 0.05, the series is stationary in the ADF sense. Adding a further first difference pushes the p-value even lower, giving an even stronger (and, per the exercise hint, standard/robust) choice. We proceed with the seasonally + first differenced series ($d=1$, $D=1$, $s=12$) for modelling below.

**h.(Basic)**	Identify a couple of ARIMA models that might be useful in describing the time series. Which of your models is the best according to their AIC values?
Hint: You can create different ARIMA models using different values of $p$ and $q$. You can either choose these values randomly(not a vey good idea) or using ACF/PACF or apply grid search.

In [ ]:
import itertools
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# Grid search SARIMA(p,1,q)(P,1,Q,12) orders on the log series (d=1, D=1 from part g)
p = q = range(0, 3)
P = Q = range(0, 2)

results = []
for order in itertools.product(p, [1], q):
    for seasonal_order in itertools.product(P, [1], Q, [12]):
        try:
            fit = SARIMAX(log_usmelec, order=order, seasonal_order=seasonal_order,
                           enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            results.append((order, seasonal_order, fit.aic))
        except Exception:
            continue

results_df = pd.DataFrame(results, columns=["order", "seasonal_order", "aic"]).sort_values("aic")
results_df.head(10)

In [ ]:
best_order, best_seasonal_order, best_aic = results_df.iloc[0]
print(f"Best model: SARIMA{best_order}x{best_seasonal_order}, AIC={best_aic:.2f}")

**i.(Regular)** Estimate the parameters of your best model and do diagnostic testing on the **residuals** (mind the difference between residual and error). Do the residuals resemble white noise? If not, try to find another ARIMA model which fits better.

Hint: Look for Ljung-Box Test and also  Shapiro-Wilk test for normality.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import shapiro

best_model = SARIMAX(log_usmelec, order=best_order, seasonal_order=best_seasonal_order,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(best_model.summary())

best_model.plot_diagnostics(figsize=(11, 8))
plt.tight_layout()
plt.show()

burn = best_order[1] + best_seasonal_order[1] * best_seasonal_order[3] + 1  # differencing (d + D*s) + filter warm-up
residuals = best_model.resid[burn:]
lb = acorr_ljungbox(residuals, lags=[12, 24], return_df=True)
print("\nLjung-Box test (H0: no residual autocorrelation):")
print(lb)

stat, p = shapiro(residuals)
print(f"\nShapiro-Wilk test for normality: stat={stat:.4f}, p-value={p:.4g}")

In [ ]:
# Inspect the largest residuals
residuals.sort_values(key=abs, ascending=False).head(5)

**Answer:** For the lowest-AIC model, SARIMA(1,1,1)(0,1,1,12), the Ljung-Box test passes at lag 12 (p≈0.32) but **rejects at lag 24 (p≈0.012)**, so its residuals are not white noise. The Shapiro-Wilk test also rejects normality (p<0.05), although mildly (W≈0.99): the largest residuals (Jan 1990, Jan 2006, ~0.09 in log units) are individual unusually cold/hot months. The residuals of the first ~14 observations are discarded, since `SARIMAX` residuals there reflect differencing/state initialisation (e.g. 0.86 in Feb 1974), not real model error.

Refitting with a richer model below, SARIMA(2,1,2)(0,1,1,12) (the best-AIC model in the grid that passes Ljung-Box at both lags), fixes the autocorrelation at a small AIC cost. We use it for the forecasts. Its Shapiro-Wilk statistic looks worse (W≈0.88), but that is driven by a single point, Mar 1974 (0.30), which is still a start-up transient of this larger model; excluding it gives W≈0.99, i.e. the same mild non-normality as before.


In [ ]:
final_model = SARIMAX(log_usmelec, order=(2, 1, 2), seasonal_order=(0, 1, 1, 12),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f"AIC={final_model.aic:.2f}")
res = final_model.resid[burn:]
print(acorr_ljungbox(res, lags=[12, 24], return_df=True))
print("Shapiro-Wilk: stat=%.4f, p-value=%.4g" % shapiro(res))
best_model = final_model


**j.(Regular)**	Forecast the next 15 years of electricity generation by the U.S. electric industry. Get the latest figures from the [EIA](https://www.eia.gov/totalenergy/data/monthly/#electricity) to check the accuracy of your forecasts.

In [ ]:
horizon = 15 * 12  # 15 years of monthly data
forecast = best_model.get_forecast(steps=horizon)
forecast_mean = np.exp(forecast.predicted_mean)                 # back-transform from log scale
forecast_ci = np.exp(forecast.conf_int(alpha=0.05))

plt.figure(figsize=(12, 5))
plt.plot(usmelec_data.index, usmelec_data['value'], label='observed')
plt.plot(forecast_mean.index, forecast_mean, label='forecast', color='red')
plt.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                  color='red', alpha=0.2, label='95% CI')
plt.title("15-year forecast of US net electricity generation")
plt.xlabel("Date"); plt.ylabel("Billion kWh")
plt.legend()
plt.show()

**Note:** To check accuracy against actual figures, download the latest monthly series from the [EIA electricity data](https://www.eia.gov/totalenergy/data/monthly/#electricity) and overlay it on the plot above for the overlapping period (2013 onward).

**k.(Regular)**	Eventually, the prediction intervals are so wide that the forecasts are not particularly useful. How many years of forecasts do you think are sufficiently accurate to be usable?

In [ ]:
rel_width = (forecast_ci.iloc[:, 1] - forecast_ci.iloc[:, 0]) / forecast_mean

plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, horizon + 1) / 12, rel_width)
plt.axhline(0.2, color='red', linestyle='--', label='20% relative width')
plt.xlabel("Forecast horizon (years)")
plt.ylabel("95% CI width / forecast")
plt.title("Growth of forecast uncertainty with horizon")
plt.legend()
plt.show()

usable_years = np.argmax(rel_width.values > 0.2) / 12
print(f"Relative CI width crosses 20% of the forecast value at ~{usable_years:.1f} years")

**Answer:** The relative width of the 95% interval grows quickly and roughly linearly with the horizon (from about 10% of the forecast value at 1 month out, to over 80% at 15 years out). Using a somewhat arbitrary but reasonable "usable" cutoff of a 20%-of-value interval width, only the first ~2 years of the forecast are usable, beyond that the interval becomes too wide (tens of percent of the forecast value) to be practically informative, even though the point forecast itself may still look reasonable further out.

## Exercise 3
Read about simple exponential smoothing in the chapter [Simple exponential smoothing](https://otexts.com/fpp3/ses.html) (FPP3, Section 8.1) and watch the associated recorded video. Then complete the following exercises in Python using the annual flow of the river Nile (1871–1970), which is included in `statsmodels`:

```python
import pandas as pd
import statsmodels.api as sm

nile = sm.datasets.nile.load_pandas().data
y = nile["volume"]
y.index = pd.PeriodIndex(nile["year"].astype(int), freq="Y")
```
**a.(Regular)** Plot the series. Explain why simple exponential smoothing is an appropriate method for this series (think about trend and seasonality).

In [ ]:
nile = sm.datasets.nile.load_pandas().data
y = nile["volume"]
y.index = pd.PeriodIndex(nile["year"].astype(int), freq="Y")

plt.figure(figsize=(10, 4))
plt.plot(y.index.to_timestamp(), y.values)
plt.title("Annual flow of the river Nile (1871-1970)")
plt.xlabel("Year")
plt.ylabel("Flow volume")
plt.show()

**Answer:** The series is annual (one observation per year), so there is no seasonality to model at all. There is a visible drop in level around 1898-99 (construction of the Aswan dam) but no consistent long-run upward or downward trend, and no seasonal cycle, the data fluctuates around a roughly constant level with random year-to-year noise. Simple exponential smoothing, which only models a level (no trend, no seasonal component), is therefore an appropriate and parsimonious method for this series.

**b.(Advanced)** Write your own function `ses_forecast(y, alpha, level)` that implements simple exponential smoothing. The arguments are `y` (the time series as a NumPy array or pandas Series), `alpha` (the smoothing parameter $\alpha$) and `level` (the initial level $\ell_0$). The function should return the one-step-ahead forecast of the next observation in the series, i.e. $\hat{y}_{T+1|T}$.

Compare your result with `statsmodels`: fit `SimpleExpSmoothing` from `statsmodels.tsa.holtwinters` with the **same** fixed $\alpha$ and $\ell_0$ (use `smoothing_level=alpha`, `initial_level=level` and `optimized=False` in `.fit()`), then call `.forecast(1)`. Does your function give the same forecast?


In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

def ses_forecast(y, alpha, level):
    """One-step-ahead SES forecast y_hat_{T+1|T} given fixed alpha and initial level l0."""
    l = level
    for obs in np.asarray(y):
        l = alpha * obs + (1 - alpha) * l
    return l

alpha, level0 = 0.2, y.iloc[0]
my_forecast = ses_forecast(y, alpha, level0)

fit_fixed = SimpleExpSmoothing(y, initialization_method="known", initial_level=level0)
fit_fixed = fit_fixed.fit(smoothing_level=alpha, optimized=False)
sm_forecast = fit_fixed.forecast(1).iloc[0]

print(f"Custom ses_forecast:      {my_forecast:.4f}")
print(f"statsmodels forecast(1):  {sm_forecast:.4f}")

**Answer:** Yes, with the same fixed $\alpha$ and $\ell_0$, the custom `ses_forecast` and `statsmodels`'s `.forecast(1)` give an identical result, confirming the recursion $\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$, $\hat{y}_{T+1|T}=\ell_T$ is implemented correctly.

**c. (Advanced)** Modify your function from the previous exercise so that it returns the sum of squared one-step errors (SSE) rather than the forecast of the next observation, i.e. `ses_sse(params, y)` where `params = [alpha, level]`.

Then use `scipy.optimize.minimize` to find the values of $\alpha$ and $\ell_0$ that minimise the SSE.

Compare your estimates with the ones `statsmodels` produces when you let it optimise the parameters itself:

```python
fit = SimpleExpSmoothing(y, initialization_method="estimated").fit()
fit.params["smoothing_level"], fit.params["initial_level"]
```

Do you get the same values? If they differ slightly, explain why (hint: different optimisers, starting values and parameter bounds).


In [ ]:
from scipy.optimize import minimize

def ses_sse(params, y):
    """Sum of squared one-step forecast errors of SES with params=[alpha, level]."""
    alpha, level = params
    l = level
    sse = 0.0
    for obs in np.asarray(y):
        err = obs - l
        sse += err ** 2
        l = alpha * obs + (1 - alpha) * l
    return sse

opt = minimize(ses_sse, x0=[0.5, y.iloc[0]], args=(y,), bounds=[(1e-4, 1 - 1e-4), (None, None)])
alpha_hat, level_hat = opt.x
print(f"Own optimisation:      alpha={alpha_hat:.4f}, level0={level_hat:.4f}")

fit_opt = SimpleExpSmoothing(y, initialization_method="estimated").fit()
print(f"statsmodels optimised:  alpha={fit_opt.params['smoothing_level']:.4f}, "
      f"level0={fit_opt.params['initial_level']:.4f}")

**Answer:** The two estimates are very close (same $\alpha \approx 0.25$, $\ell_0$ differs by only ~0.01) but not bit-for-bit identical. This is expected: `scipy.optimize.minimize` (default L-BFGS-B here, with explicit bounds and starting values `[0.5, y[0]]`) and `statsmodels`'s internal optimiser use different starting points, bounds/parameterisations and convergence tolerances, so they stop at slightly different points on what is a very flat SSE surface near the minimum. The same tiny gap carries over to the forecast in (d) (805.315 vs 805.317).

**d. (Advanced)** Combine your previous two functions into a single function `ses(y)` that both finds the optimal values of $\alpha$ and $\ell_0$ **and** returns the forecast of the next observation in the series. Check that its output matches `.forecast(1)` from the optimised `statsmodels` model.

In [ ]:
def ses(y):
    """Fit SES by minimising SSE, then return the one-step-ahead forecast."""
    y = np.asarray(y)
    opt = minimize(ses_sse, x0=[0.5, y[0]], args=(y,), bounds=[(1e-4, 1 - 1e-4), (None, None)])
    alpha_hat, level_hat = opt.x
    forecast = ses_forecast(y, alpha_hat, level_hat)
    return forecast, alpha_hat, level_hat

my_forecast, alpha_hat, level_hat = ses(y)
sm_forecast = fit_opt.forecast(1).iloc[0]

print(f"ses(y) forecast:                 {my_forecast:.4f} (alpha={alpha_hat:.4f}, level0={level_hat:.4f})")
print(f"statsmodels optimised forecast:  {sm_forecast:.4f}")